# AmSC IRI Multi-Site Job Submission

AmSC Resource Orchestration Toolkit (AmSCROT) - Orchestrating Infrastructure Service capabilities.

This notebook demonstrates how to submit compute jobs to **ESnet IRI** and **NERSC IRI** sites using the `amscrot` Client module.

## Workflow Overview

1. **Initialize** the AmSCROT client
2. **Create** a session
4. **Set up** ESnet and NERSC IRI Service Clients
5. **Discover** available compute resources at each site
6. **Define** and submit batch jobs
7. **Monitor** job status
8. **Clean up**

## Prerequisites

### Required Packages
- `amscrot-py` (installed)
- `globus-sdk` (installed)
- `dotenv` (installed)

### Installation

```
pip install amscrot-py globus-sdk dotenv
```

### Credentials

IRI service clients authenticate via API keys stored in `~/.amscrot/credentials.yml`. You need entries for each site that will be used for Job submission.

```yaml
# ~/.amscrot/credentials.yml

esnet-iri-east:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://iri-dev.ppg.es.net

esnet-iri-west:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://esnet-west.sdn-sense.net

nersc-iri:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://api.iri.nersc.gov
```

---
## 0. Use Globus Auth Token

In [ ]:
import globus_sdk
import datetime
import time
import os
import yaml
from dotenv import load_dotenv


load_dotenv()  # take environment variables from .env file
GLOBUS_ID = os.getenv("GLOBUS_ID")
GLOBUS_SECRET = os.getenv("GLOBUS_SECRET")

print ("Using Globus ID", GLOBUS_ID)

# Create a confidential client
client = globus_sdk.ConfidentialAppAuthClient(GLOBUS_ID, GLOBUS_SECRET)
# Start the OAuth flow
client.oauth2_start_flow(
    redirect_uri="http://localhost:5000/callback",  # or your registered redirect URI
    requested_scopes=["openid", "profile", "email", "urn:globus:auth:scope:auth.globus.org:view_identities"]
)
# Get the authorization URL
authorize_url = client.oauth2_get_authorize_url()
print(f"Visit this URL in your browser:\n{authorize_url}\n")

# After visiting the URL and authorizing, you'll be redirected to a URL with a code parameter
auth_code = input("Paste the 'code' parameter from the redirect URL: ")

# Exchange the code for tokens

token_response = client.oauth2_exchange_code_for_tokens(auth_code)
access_token_data = token_response.by_resource_server['auth.globus.org']
access_token = access_token_data['access_token']
expires_at = access_token_data['expires_at_seconds']

# Convert to human-readable time
expiration_time = datetime.datetime.fromtimestamp(expires_at)
print(f"Token expires at: {expiration_time}")

# Calculate how long until expiration
seconds_until_expiration = expires_at - time.time()
hours_until_expiration = seconds_until_expiration / 3600

print(f"Token expires in {hours_until_expiration:.2f} hours")
# print("---------------------")
# print(access_token_data)
# print("---------------------")
print(f"\nYour access token:\n{access_token}")
# auth_client = globus_sdk.AuthClient(authorizer=globus_sdk.AccessTokenAuthorizer(access_token))
# userinfo = auth_client.userinfo()
# print(f"\nYour user information:\n{userinfo}")

# Create credentials data
credentials = {
    'esnet-iri-east': {
        'api_key': access_token,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'api_key': access_token,
        'api_endpoint': 'https://esnet-west.sdn-sense.net'
    },
    'nersc-iri': {
        'api_key': access_token,
        'api_endpoint': 'https://api.iri.nersc.gov'
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 1. Initialize Client & Session

In [ ]:
import time
from amscrot.client.client import Client
from amscrot.client.job import Job, JobType, JobServiceType, JobSpec, JobState
from amscrot.serviceclient import ServiceClient
from amscrot.util.constants import Constants

client = Client()
session = client.create_session("iri-multsite-jobs")
print("Client and session initialized.")

## 3. Set Up IRI Service Clients

We create three service clients — one for each IRI site. Each client loads its credentials from the corresponding profile in `~/.amscrot/credentials-new.yml`.

In [ ]:
east_client = ServiceClient.create(
    type=Constants.ServiceType.ESNET_IRI,
    name="iri-east",
    profile="esnet-iri-east",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(east_client)

west_client = ServiceClient.create(
    type=Constants.ServiceType.ESNET_IRI,
    name="iri-west",
    profile="esnet-iri-west",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(west_client)

nersc_client = ServiceClient.create(
    type=Constants.ServiceType.NERSC_IRI,
    name="nersc-iri",
    profile="nersc-iri",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(nersc_client)

## 4. Discover Compute Resources

Each service client's `discover()` method returns a `DiscoveryResult` container with typed accessors for each resource type. We use `.compute` to find available compute resources at each site.

In [ ]:
east_discovery = east_client.discover()
# west_discovery = west_client.discover()
nersc_discovery = nersc_client.discover()

print(f"ESnet East discovery: {east_discovery.summary()}")
# print(f"ESnet West discovery: {west_discovery.summary()}")
print(f"NERSC discovery: {nersc_discovery.summary()}")

assert east_discovery.compute, "No compute resources found on East site!"
# assert west_discovery.compute, "No compute resources found on West site!"
assert nersc_discovery.compute, "No compute resources found on NERSC site!"

# For ESnet, just grab the first compute resource
east_resource_id = east_discovery.compute[0].data.get("id")
# west_resource_id = west_discovery.compute[0].data.get("id")
compute_resources = nersc_discovery.compute

# For NERSC, locate the compute resource with group "perlmutter" and named "compute"
target_resource = None
for resource in compute_resources:
    data = resource.data
    if data.get('group') == 'perlmutter' and data.get('name') == 'compute':
        target_resource = resource
        break

if not target_resource:
    self.skipTest("Target compute resource (perlmutter/compute) not found.")

nersc_resource_data = target_resource.data
nersc_resource_id = nersc_resource_data.get('id')

if nersc_resource_id:
    print(f"Found target NERSC compute resource: {nersc_resource_data}")

print(f"\nEast resource_id: {east_resource_id}")
# print(f"West resource_id: {west_resource_id}")
print(f"NERSC resource_id: {nersc_resource_id}")

## 5. Define Job Specs & Jobs

Each job is a simple batch job that runs `/bin/echo` on the discovered compute resource. The `resource_id` is set dynamically from the discovery step above.

In [ ]:
common_resources = {
    "node_count": 1,
    "process_count": 1,
    "processes_per_node": 1,
    "cpu_cores_per_process": 1,
    "gpu_cores_per_process": None,
    "exclusive_node_use": False,
    "memory": 268435456
}

esnet_attributes = {
    "directory": "/tmp",
    "duration": 600,
    "queue_name": "debug",
    "account": "interactive"
}

nersc_attributes = {
    "directory": "/tmp",
    "duration": 600,
    "queue_name": "debug",
    "account": "amsc013",
    "pre_launch": ""
}

spec_east = JobSpec(
    executable=["/bin/echo", "Hello AmSC East"],
    resources=common_resources,
    attributes={"resource_id": east_resource_id, **esnet_attributes}
)

# spec_west = JobSpec(
#     executable=["/bin/echo", "Hello AmSC West"],
#     resources=common_resources,
#     attributes={"resource_id": west_resource_id, **nersc_attributes}
# )

spec_nersc = JobSpec(
    executable=["/bin/echo", "Hello AmSC NERSC"],
    resources=common_resources,
    attributes={"resource_id": nersc_resource_id, **nersc_attributes}
)

job1 = Job(name="job-1", type=JobType.COMPUTE,
            service_type=JobServiceType.BATCH,
            service_client=east_client,
            job_spec=spec_east)

# job2 = Job(name="job-2", type=JobType.COMPUTE,
#             service_type=JobServiceType.BATCH,
#             service_client=west_client,
#             job_spec=spec_west)

job3 = Job(name="job-3", type=JobType.COMPUTE,
            service_type=JobServiceType.BATCH,
            service_client=nersc_client,
            job_spec=spec_nersc)

session.add_job(job1)
#session.add_job(job2)
session.add_job(job3)

print("Jobs defined and added to session.")

## 6. Plan

The plan phase validates all resources and job specs before anything is created.

In [ ]:
try:
    session.plan()
except Exception as e:
    print(f"Failed to plan jobs: {e}")

## 7. Apply and monitor

Apply creates resources (if networking is enabled) and submits the compute jobs.
`session.wait()` then polls both jobs until they complete (or raise `WaitTimeoutError` after 120 seconds).

In [ ]:
try:
    rc = session.apply()
    if rc:
        raise RuntimeError(f"Session apply failed with rc={rc}")
except Exception as e:
    print(f"Failed to apply session: {e}")
    raise

print("Session applied successfully.")

results = session.wait(
    # jobs=[job1, job2, job3],
    jobs = [job1, job3],
    target_states=[JobState.COMPLETED, JobState.FAILED, JobState.CANCELED],
    timeout=600,
    interval=2,
    verbose=True,
)

s1 = results["job-1"]
# s2 = results["job-2"]
s3 = results["job-3"]

assert s1.state == JobState.COMPLETED, f"Job1 failed or timed out: {s1}"
# assert s2.state == JobState.COMPLETED, f"Job2 failed or timed out: {s2}"
assert s3.state == JobState.COMPLETED, f"Job3 failed or timed out: {s3}"

print(f"\n✅ All jobs completed successfully!")

## 9. Clean Up

Destroy the session to tear down any provisioned resources and cancel remaining jobs.

In [ ]:
session.destroy()